# 04 — Embeddings and Linear Layers

## Goal

Token IDs are discrete categorical identifiers.

For example,

$$
3,\ 7,\ 12
$$

do not contain meaningful numerical relationships by themselves.

A neural network therefore needs to map each token ID to a continuous
vector representation.

If the vocabulary size is

$$
V
$$

and the embedding dimension is

$$
C,
$$

we store an embedding table

$$
E \in \mathbb{R}^{V \times C}.
$$

Each row corresponds to one token in the vocabulary.

Looking up token ID $i$ returns row $i$ of the embedding table:

$$
E[i] \in \mathbb{R}^{C}.
$$

For a batch of token IDs with shape

$$
(B,T),
$$

embedding lookup produces

$$
(B,T,C).
$$

This is the first transition from discrete token IDs to continuous
floating-point representations.

In [1]:
import torch

In [2]:
vocab_size: int = 6
embedding_dim: int = 4

embedding_table: torch.Tensor = torch.randn(
    vocab_size,
    embedding_dim,
)

print(embedding_table)
print("shape:", embedding_table.shape)

tensor([[ 2.2755,  0.7350,  1.7656,  0.5304],
        [ 1.0055,  0.5170,  0.7333,  1.5948],
        [ 1.9641, -0.7612,  0.4898, -1.0264],
        [ 0.2576, -1.7156, -0.8795,  1.7505],
        [ 0.2865, -0.6664,  2.1062,  0.7725],
        [-1.4268,  0.0311, -1.4605,  0.4145]])
shape: torch.Size([6, 4])


In [3]:
token_ids: torch.Tensor = torch.tensor(
    [2, 4, 1],
    dtype=torch.long,
)

embeddings: torch.Tensor = embedding_table[token_ids]

print("token IDs:", token_ids)
print("embeddings:")
print(embeddings)

print("token shape:", token_ids.shape)
print("embedding shape:", embeddings.shape)

token IDs: tensor([2, 4, 1])
embeddings:
tensor([[ 1.9641, -0.7612,  0.4898, -1.0264],
        [ 0.2865, -0.6664,  2.1062,  0.7725],
        [ 1.0055,  0.5170,  0.7333,  1.5948]])
token shape: torch.Size([3])
embedding shape: torch.Size([3, 4])


In [4]:
token_batch: torch.Tensor = torch.tensor(
    [
        [1, 2, 3],
        [4, 2, 0],
    ],
    dtype=torch.long,
)

token_embeddings: torch.Tensor = embedding_table[token_batch]

print("token batch:")
print(token_batch)

print("\nembeddings:")
print(token_embeddings)

print("\ntoken shape:", token_batch.shape)
print("embedding shape:", token_embeddings.shape)

token batch:
tensor([[1, 2, 3],
        [4, 2, 0]])

embeddings:
tensor([[[ 1.0055,  0.5170,  0.7333,  1.5948],
         [ 1.9641, -0.7612,  0.4898, -1.0264],
         [ 0.2576, -1.7156, -0.8795,  1.7505]],

        [[ 0.2865, -0.6664,  2.1062,  0.7725],
         [ 1.9641, -0.7612,  0.4898, -1.0264],
         [ 2.2755,  0.7350,  1.7656,  0.5304]]])

token shape: torch.Size([2, 3])
embedding shape: torch.Size([2, 3, 4])


### Shape Interpretation

Before embedding lookup:

$$
X.shape = (B,T).
$$

Each element is an integer token ID.

After embedding lookup:

$$
H.shape = (B,T,C).
$$

The dimensions now mean:

<pre>
dim 0 → batch
dim 1 → token position
dim 2 → embedding features
</pre>

For example,

`H[b, t]`

is the embedding vector for token position `t` in batch example `b`.

Its shape is

$$
(C).
$$

And

`H[b, t, c]`

is one scalar feature of that embedding vector.

In [5]:
print(
    torch.equal(
        token_embeddings[0, 1],
        embedding_table[token_batch[0, 1]],
    )
)

True


## 4. Embedding Lookup and One-Hot Vectors

Embedding lookup can be understood as a special case of matrix
multiplication.

Suppose the vocabulary size is

$$
V
$$

and the embedding dimension is

$$
C.
$$

The embedding table is

$$
E \in \mathbb{R}^{V \times C}.
$$

For token ID $i$, we can construct a one-hot vector

$$
o_i \in \mathbb{R}^{V},
$$

where every element is zero except position $i$.

For example, if

$$
V = 6
$$

and the token ID is

$$
i = 2,
$$

then

$$
o_i =
[0, 0, 1, 0, 0, 0].
$$

Multiplying the one-hot vector by the embedding matrix selects row $i$:

$$
o_i E = E[i].
$$

Therefore, embedding lookup is mathematically equivalent to one-hot
matrix multiplication, but direct indexing is much more efficient
because it avoids constructing a large sparse one-hot vector.

In [6]:
token_id: int = 2

one_hot: torch.Tensor = torch.zeros(vocab_size)
one_hot[token_id] = 1.0

embedding_from_matmul: torch.Tensor = one_hot @ embedding_table
embedding_from_lookup: torch.Tensor = embedding_table[token_id]

print("one-hot:", one_hot)
print("matmul:", embedding_from_matmul)
print("lookup:", embedding_from_lookup)

print(
    "same:",
    torch.allclose(
        embedding_from_matmul,
        embedding_from_lookup,
    ),
)

one-hot: tensor([0., 0., 1., 0., 0., 0.])
matmul: tensor([ 1.9641, -0.7612,  0.4898, -1.0264])
lookup: tensor([ 1.9641, -0.7612,  0.4898, -1.0264])
same: True


In [7]:
embedding_table.requires_grad_(True)

token_ids: torch.Tensor = torch.tensor(
    [1, 2, 1],
    dtype=torch.long,
)

embeddings: torch.Tensor = embedding_table[token_ids]

loss: torch.Tensor = embeddings.sum()

loss.backward()

print(embedding_table.grad)

tensor([[0., 0., 0., 0.],
        [2., 2., 2., 2.],
        [1., 1., 1., 1.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]])


### Gradient Flow Through an Embedding Table

Embedding lookup does not update the entire vocabulary equally.

Only embedding rows that are used during the forward pass receive
non-zero gradients.

If the same token appears multiple times, gradient contributions from
all occurrences accumulate into the same embedding vector.

This is another example of gradient accumulation through multiple paths
in the computation graph.

## 5. Linear Layers

A linear layer transforms a vector from one feature space into another.

Given

$$
x \in \mathbb{R}^{C_{\text{in}}},
$$

a linear layer computes

$$
y = xW + b,
$$

where

$$
W \in \mathbb{R}^{C_{\text{in}} \times C_{\text{out}}}
$$

and

$$
b \in \mathbb{R}^{C_{\text{out}}}.
$$

The output therefore has shape

$$
y \in \mathbb{R}^{C_{\text{out}}}.
$$

For language models, the same transformation is usually applied
independently to every token vector in every batch example.

In [8]:
input_dim: int = 4
output_dim: int = 3

weight: torch.Tensor = torch.randn(
    input_dim,
    output_dim,
    requires_grad=True,
)

bias: torch.Tensor = torch.randn(
    output_dim,
    requires_grad=True,
)

In [9]:
x: torch.Tensor = torch.randn(input_dim)

y: torch.Tensor = x @ weight + bias

print("x shape:", x.shape)
print("weight shape:", weight.shape)
print("bias shape:", bias.shape)
print("y shape:", y.shape)

x shape: torch.Size([4])
weight shape: torch.Size([4, 3])
bias shape: torch.Size([3])
y shape: torch.Size([3])


In [10]:
batch_size: int = 2
context_length: int = 3
input_dim: int = 4
output_dim: int = 6

x: torch.Tensor = torch.randn(
    batch_size,
    context_length,
    input_dim,
)

weight: torch.Tensor = torch.randn(
    input_dim,
    output_dim,
)

bias: torch.Tensor = torch.randn(output_dim)

y: torch.Tensor = x @ weight + bias

print("x:", x.shape)
print("weight:", weight.shape)
print("bias:", bias.shape)
print("y:", y.shape)

x: torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
bias: torch.Size([6])
y: torch.Size([2, 3, 6])


## 6. A Minimal Linear Layer

A linear layer applies the same affine transformation to every input
vector:

$$
y = xW + b.
$$

If

$$
x \in \mathbb{R}^{C_{\text{in}}},
$$

then we choose

$$
W \in \mathbb{R}^{C_{\text{in}} \times C_{\text{out}}}
$$

and

$$
b \in \mathbb{R}^{C_{\text{out}}}.
$$

The result is therefore

$$
y \in \mathbb{R}^{C_{\text{out}}}.
$$

For an input tensor of shape

$$
(B,T,C_{\text{in}}),
$$

the same matrix is applied independently to every token vector:

$$
(B,T,C_{\text{in}})
\rightarrow
(B,T,C_{\text{out}}).
$$

The batch and sequence dimensions are preserved.

In [11]:
class Linear:
    def __init__(self, in_features: int, out_features: int) -> None:
        self.weight: torch.Tensor = torch.randn(
            in_features, out_features, requires_grad=True
        )

        self.bias: torch.Tensor = torch.zeros(out_features, requires_grad=True)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.weight + self.bias

    def parameters(self) -> list[torch.Tensor]:
        return [self.weight, self.bias]


layer = Linear(
    in_features=4,
    out_features=6,
)

x: torch.Tensor = torch.randn(
    2,
    3,
    4,
)

y: torch.Tensor = layer(x)

print("x:", x.shape)
print("weight:", layer.weight.shape)
print("bias:", layer.bias.shape)
print("y:", y.shape)

x: torch.Size([2, 3, 4])
weight: torch.Size([4, 6])
bias: torch.Size([6])
y: torch.Size([2, 3, 6])


## 7. Comparing with `torch.nn.Linear`

Our manual implementation stores the weight matrix as

$$
W_{\text{manual}}
\in
\mathbb{R}^{C_{\text{in}} \times C_{\text{out}}}
$$

and computes

$$
xW_{\text{manual}} + b.
$$

PyTorch's `nn.Linear` stores its weight using the opposite orientation:

$$
W_{\text{torch}}
\in
\mathbb{R}^{C_{\text{out}} \times C_{\text{in}}}.
$$

Its forward computation can therefore be written as

$$
xW_{\text{torch}}^T + b.
$$

These are mathematically equivalent representations.

The difference is only the convention used to store the weight matrix.

In [12]:
import torch.nn as nn


torch_layer = nn.Linear(
    in_features=4,
    out_features=6,
)

print(torch_layer.weight.shape)
print(torch_layer.bias.shape)

torch.Size([6, 4])
torch.Size([6])


In [13]:
with torch.no_grad():
    torch_layer.weight.copy_(layer.weight.T)

    torch_layer.bias.copy_(layer.bias)

manual_output: torch.Tensor = layer(x)
torch_output: torch.Tensor = torch_layer(x)

print(
    torch.allclose(
        manual_output,
        torch_output,
    )
)

True


## 8. Embedding Lookup vs Linear Transformation

Embedding and linear layers both contain learnable matrices, but they
perform fundamentally different operations.

### Embedding lookup

An embedding layer receives an integer token ID:

$$
i
$$

and selects one row from an embedding table:

$$
E[i].
$$

For token IDs with shape

$$
(B,T),
$$

the output has shape

$$
(B,T,C).
$$

No weighted combination of all embedding rows is required.

### Linear transformation

A linear layer receives an already continuous feature vector

$$
x \in \mathbb{R}^{C_{\text{in}}}
$$

and mixes all input features through matrix multiplication:

$$
y = xW + b.
$$

Therefore:

<pre>
Embedding:
integer index
    ↓
select one row

Linear:
continuous vector
    ↓
mix features through matrix multiplication
</pre>

In [14]:
layer = Linear(
    in_features=3,
    out_features=2,
)

x: torch.Tensor = torch.tensor([1.0, 2.0, 3.0])

y: torch.Tensor = layer(x)

loss: torch.Tensor = y.sum()

loss.backward()

print("weight grad:")
print(layer.weight.grad)

print("bias grad:")
print(layer.bias.grad)

weight grad:
tensor([[1., 1.],
        [2., 2.],
        [3., 3.]])
bias grad:
tensor([1., 1.])


## 9. Parameter Counting

For a linear transformation

$$
C_{\text{in}}
\rightarrow
C_{\text{out}},
$$

the weight matrix contains

$$
C_{\text{in}} C_{\text{out}}
$$

parameters.

If a bias is used, it adds

$$
C_{\text{out}}
$$

more parameters.

Therefore,

$$
N_{\text{parameters}}
=
C_{\text{in}}C_{\text{out}}
+
C_{\text{out}}.
$$